Import

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel, Matern
from scipy.stats import norm
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("Libraries loaded successfully!")

In [ ]:
# Function metadata
FUNCTIONS = {
    1: {"dims": 2, "initial_points": 10, "analogy": "F1"},
    2: {"dims": 2, "initial_points": 10, "analogy": "F2"},
    3: {"dims": 3, "initial_points": 15, "analogy": "F3"},
    4: {"dims": 4, "initial_points": 30, "analogy": "F4"},
    5: {"dims": 4, "initial_points": 20, "analogy": "F5"},
    6: {"dims": 5, "initial_points": 20, "analogy": "F6"},
    7: {"dims": 6, "initial_points": 30, "analogy": "F7"},
    8: {"dims": 8, "initial_points": 40, "analogy": "F8"},
}

# Display function info
func_df = pd.DataFrame([
    {"Function": f"F{k}", **v} for k, v in FUNCTIONS.items()
])
print("Function Overview:")
display(func_df)

Load Initial Data

Initial .npy data files are provided by Imperial College Business School via the course portal. They are not stored in this repository due to licensing issue.

In [ ]:
def load_function_data(func_num, data_dir="../data/raw"):
    """Load data for a specific function."""
    import os
    func_dir = os.path.join(data_dir, f"function_{func_num}")
    X = np.load(os.path.join(func_dir, "initial_inputs.npy"))
    Y = np.load(os.path.join(func_dir, "initial_outputs.npy"))
    return X, Y.ravel()

Gaussian Process Surrogate Model

Willb using  a Gaussian Process with RBF kernel to approximate the unknown functions.

In [ ]:
def create_gp_surrogate(kernel_type="rbf", length_scale=0.5, noise_level=0.2):
    """
    Create a Gaussian Process surrogate model.
    
    Args:
        kernel_type: 'rbf' or 'matern'
        length_scale: Initial length scale
        noise_level: Expected noise level
        
    Returns:
        Configured GaussianProcessRegressor
    """
    if kernel_type == "rbf":
        base_kernel = RBF(length_scale=length_scale)
    else:
        base_kernel = Matern(length_scale=length_scale, nu=2.5)
    
    kernel = ConstantKernel(1.0) * base_kernel + WhiteKernel(noise_level=noise_level)
    
    gp = GaussianProcessRegressor(
        kernel=kernel,
        n_restarts_optimizer=10,
        normalize_y=True,
        random_state=42
    )
    
    return gp

print("GP Surrogate model defined.")

Acquisition Functions

I will be implementing three acquisition functions to guide query selection:

UCB (Upper Confidence Bound): Balances exploration and exploitation via β parameter
EI (Expected Improvement): Maximises expected gain over current best
PI (Probability of Improvement): Maximises probability of beating current best

In [ ]:
def upper_confidence_bound(X, gp, beta=2.0):
    """UCB(x) = μ(x) + β * σ(x)"""
    mean, std = gp.predict(X, return_std=True)
    return mean + beta * std


def expected_improvement(X, gp, y_best, xi=0.01):
    """EI(x) = E[max(f(x) - y_best - ξ, 0)]"""
    mean, std = gp.predict(X, return_std=True)
    std = np.maximum(std, 1e-9)
    z = (mean - y_best - xi) / std
    ei = (mean - y_best - xi) * norm.cdf(z) + std * norm.pdf(z)
    ei[std < 1e-9] = 0.0
    return ei


def probability_of_improvement(X, gp, y_best, xi=0.01):
    """PI(x) = P(f(x) > y_best + ξ)"""
    mean, std = gp.predict(X, return_std=True)
    std = np.maximum(std, 1e-9)
    z = (mean - y_best - xi) / std
    return norm.cdf(z)


def format_query(x):
    """Format query for portal submission."""
    x = np.clip(x, 0.0, 0.999999)
    return "-".join([f"{val:.6f}" for val in x])


print("Acquisition functions defined.")

Complete Results (Iterations 1 to 13)

Below is the complete query history and results from all 13 rounds of optimisation.

In [ ]:
results_df = pd.read_csv("../results/BBO Challenege.csv")
print(f"Total observations: {len(results_df)}")
display(results_df.head(20))

In [ ]:
final_results = {
    "Function": ["F1", "F2", "F3", "F4", "F5", "F6", "F7", "F8"],
    "Dimensions": [2, 2, 3, 4, 4, 5, 6, 8],
    "Initial Best": [1.03, 0.131, -0.063, -2.24, 21, -1.28, 0.007, 6.34],
    "Final Round": [1.55, 0.158, -0.076, -2.24, 21, -1.317, 0.007, 6.34],
    "Improvement": ["N", "N", "N", "Y", "Y", "N", "Y", "Y"],
    "Strategy": ["Diagonal discovery", "Region refinement", "Gradient following",
                 "Consistent gradient", "Boundary pushing", "Revert-to-best",
                 "Small perturbations", "Plateau refinement"],
    "Best Round": ["R1", "R5", "R2", "R13", "R13", "R11", "R13", "R13"]
}

final_df = pd.DataFrame(final_results)
print("\n" + "="*80)
print("FINAL RESULTS SUMMARY (After Round 13)")
print("="*80)
display(final_df)

In [ ]:
best_queries = {
    "F1": {"query": "0.123254-0.247652", "output": 1.035, "round": 1},
    "F2": {"query": "0.190000-0.220000", "output": 0.131, "round": 5},
    "F3": {"query": "0.700000-0.300000-0.600000", "output": -0.076, "round": 2},
    "F4": {"query": "0.340000-0.280000-0.330000-0.280000", "output": -2.2401, "round": 13},
    "F5": {"query": "0.500000-0.460000-0.540000-0.560000", "output": 21.00, "round": 13},
    "F6": {"query": "0.650000-0.320000-0.660000-0.320000-0.650000", "output": -0.128, "round": 11},
    "F7": {"query": "0.750000-0.730000-0.750000-0.730000-0.750000-0.730000", "output": 0.007, "round": 13},
    "F8": {"query": "0.638000-0.442000-0.986000-0.128000-0.528000-0.178000-0.774000-0.734000", 
           "output": 6.335, "round": 13},
}

print("\n" + "="*80)
print("BEST QUERIES PER FUNCTION")
print("="*80)
for func, data in best_queries.items():
    print(f"\n{func} (Round {data['round']}):")
    print(f"  Query:  {data['query']}")
    print(f"  Output: {data['output']}")

Convergence Analysis

Visualise how the best output evolved over the 13 rounds.

In [ ]:
convergence_data = {
    "F1": [1.035, 2.855, 4.279, 1.074, 4.014, 2.675, 3.610, 6.790, 7.321, 3.681, 2.812, 8.358, 1.556],
    "F2": [-0.114, 0.287, 0.154, -0.188, 0.131, 0.194, 0.269, 0.102, 0.052, -0.026, 0.164, 0.016, 0.157 ],
    "F3": [-0.148, -0.064,	-0.094,	-0.087,	-0.069,	-0.074,	-0.082,	-0.098,	-0.093,	-0.069,	-0.068,	-0.072,	-0.076],
    "F4": [-10.873,	-27.402, -35.012, -39.923, -47.843,	-10.584, -9.772, -8.988, -8.284, -7.676, -6.562, -6.139, -2.241],
    "F5": [159.94, 159.435,	159.724, 136.851, 118.608, 159.823,	159.759, 159.204, 158.593, 156.995, 151.502, 138.233, 21.003],
    "F6": [-1.574, -1.398, -1.559, -1.391, -1.623, -1.369, -1.398, -1.389, -1.306, -1.294, -1.286, -1.392, -1.317],
    "F7": [0.302, 0.105, 0.253, 0.152, 0.215, 0.181, 0.198,	0.047, 0.039, 0.021, 0.011,	0.01, 0.007],
    "F8": [7.109, 7.07, 7.034, 7.124, 7.034, 7.018, 6.985, 6.88, 6.796, 6.741, 7.396, 7.351, 6.336],
}

rounds = ["R1", "R2", "R3", "R4", "R5", "R6", "R7", "R8", "R9", "R10", "R11", "R12", "R13"]

In [ ]:
# Plot convergence for all functions
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

colors = ['#2ecc71', '#3498db', '#9b59b6', '#e74c3c', '#f39c12', '#1abc9c', '#34495e', '#e67e22']

for i, (func, values) in enumerate(convergence_data.items()):
    ax = axes[i]
    
    ax.plot(range(len(values)), values, '-o', color=colors[i], linewidth=2, markersize=6)
    ax.fill_between(range(len(values)), values, alpha=0.2, color=colors[i])
    
    # Highlight best round
    best_idx = np.argmax(values)
    ax.scatter([best_idx], [values[best_idx]], color='red', s=150, marker='*', zorder=5, 
               label=f'Best: {rounds[best_idx]}')
    
    ax.set_title(f"{func} ({FUNCTIONS[i+1]['dims']}D)", fontsize=12, fontweight='bold')
    ax.set_xlabel("Round")
    ax.set_ylabel("Best Output")
    ax.set_xticks(range(0, 14, 2))
    ax.set_xticklabels([rounds[j] for j in range(0, 14, 2)], rotation=45)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='lower right', fontsize=8)

plt.suptitle("Optimisation Convergence: Best Output per Round (13 Rounds)", 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("../results/figures/convergence_final.png", dpi=150, bbox_inches='tight')
plt.show()

print("Convergence plot saved to results/figures/convergence_final.png")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# F5: Outstanding improvement
ax1 = axes[0]
ax1.plot(range(14), convergence_data["F5"], 'g-o', linewidth=2, markersize=8)
ax1.fill_between(range(14), convergence_data["F5"], alpha=0.3, color='green')
ax1.set_title("F5: Outstanding", fontsize=12, fontweight='bold')
ax1.set_xlabel("Round")
ax1.set_ylabel("Output")
ax1.annotate(f'3,934', xy=(13, 3934), xytext=(10, 3500),
             arrowprops=dict(arrowstyle='->', color='red'), fontsize=10, color='red')

# F4: Nearest to zero
ax2 = axes[1]
ax2.plot(range(14), convergence_data["F4"], 'b-o', linewidth=2, markersize=8)
ax2.fill_between(range(14), convergence_data["F4"], alpha=0.3, color='blue')
ax2.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Zero line')
ax2.set_title("F4: Nearest to zero (-10.87 → -2.24)", fontsize=12, fontweight='bold')
ax2.set_xlabel("Round")
ax2.set_ylabel("Output")
ax2.legend()


# F7: Optimal breakthrough
ax3 = axes[2]
f7_values = [0.302, 0.105, 0.253, 0.152, 0.215, 0.181, 0.198, 0.047, 0.039, 0.021, 0.011, 0.01, 0.007]
ax3.semilogy(range(14), [max(v, 1e-12) for v in f1_values], 'r-o', linewidth=2, markersize=8)
ax3.set_title("F7: Optimal Breakthrough (Log Scale)", fontsize=12, fontweight='bold')
ax3.set_xlabel("Round")
ax3.set_ylabel("Output (log scale)")
ax3.annotate('Breakthrough!', xy=(11, 1.4e-4), xytext=(8, 1e-3),
             arrowprops=dict(arrowstyle='->', color='red'), fontsize=10, color='red')

plt.tight_layout()
plt.savefig("../results/figures/breakthroughs.png", dpi=150, bbox_inches='tight')
plt.show()
ax3.semilogy(range(14), [max(v, 1e-12) for v in f1_values], 'r-o', linewidth=2, markersize=8)
ax3.set_title("F7: Optimal Breakthrough (Log Scale)", fontsize=12, fontweight='bold')
ax3.set_xlabel("Round")
ax3.set_ylabel("Output (log scale)")
ax3.annotate('Breakthrough!', xy=(11, 1.4e-4), xytext=(8, 1e-3),
             arrowprops=dict(arrowstyle='->', color='red'), fontsize=10, color='red')

plt.tight_layout()
plt.savefig("../results/figures/breakthroughs.png", dpi=150, bbox_inches='tight')
plt.show()

Key Insights & Lessons Learned

Strategy Evolution

During the exploration phase, between round 1 to 3 I was getting a feel of the boundary so explotation proved more realiable. At round 4 to 6 I was estimating the gradient and starting to get a directional inference. F3/F4 were developing clear gradients. Round 7 to 9 was the refinement stage where I had enough data to know when to abandon the failed exploration. The final rounds of 10 to 13 was agressive explotation with F4 and F5 being eveident of boundary pushing. 

In Conclusion, after 13 rounds of Bayesian optimisation across eight black-box functions:

Outstanding Results

F5 showed significant imporvement through systematic boundary pushing
F4 showed sign of to an optimal score in the later rounds through consistent gradient following
F7: showed optimal breakthrough from high to low output

Challenges

F2, F3, F1: Stochastic behaviour limited optimisation potential
F6, F8: High dimensionality led to plateau effects
Core Insight

Overall the Black-box optimisation is about pattern recognition under uncertainty. The technical tools (GPs, acquisition functions) matter less than developing intuition for when to explore versus exploit, when to persist versus pivot, and when to accept what is "good enough."